# Previsão de acidentes graves em rodovias federais brasileiras com dados abertos da PRF

**Projeto Final — Inteligência Artificial (UFPI) — Prof. Dr. Pedro Almir Martins de Oliveira**

| | |
|---|---|
| **Autor** | *PREENCHER: nome completo e matrícula* |
| **Objetivo** | Avaliar se é possível prever, com as informações de local, horário e condições da via, se um acidente em rodovia federal terá vítima grave ou fatal. |
| **Pergunta de pesquisa** | Quanto da gravidade de um acidente é previsível apenas pelo contexto (onde, quando e em que condições ele ocorre), e quanto do desempenho relatado em trabalhos semelhantes depende de informações que só existem depois do acidente? |
| **Tipo de aprendizado** | Supervisionado — classificação binária (acidente *grave* × *não grave*). |
| **Dados** | Dados Abertos da Polícia Rodoviária Federal (PRF), acidentes agrupados por ocorrência, 2021–2025. |

Este notebook é ao mesmo tempo o código e o relatório técnico do projeto. Ele baixa os dados direto da fonte pública e executa do início ao fim sem arquivos locais.

## 1. Descrição do problema

**Contexto.** A PRF registra dezenas de milhares de acidentes por ano nas rodovias federais. Uma parte deles deixa mortos ou feridos graves, e é nesses casos que o tempo de resposta do socorro e a fiscalização preventiva mais importam.

**Decisão apoiada.** Um modelo que estime o risco de um acidente ser grave a partir do contexto (trecho, horário, clima, tipo de pista) pode apoiar duas decisões: onde e quando concentrar fiscalização e equipes de atendimento, e que prioridade dar a uma ocorrência recém-comunicada, antes de a equipe chegar ao local.

**Público beneficiado.** Planejamento operacional da PRF, serviços de atendimento de emergência (SAMU, bombeiros) e concessionárias de rodovias.

**Limites do escopo.**
- A base só contém acidentes que **aconteceram e foram registrados**. O modelo estima a gravidade *dado que houve acidente*; ele não estima a probabilidade de um acidente ocorrer.
- Só rodovias federais. Vias estaduais e urbanas ficam de fora.
- O modelo encontra associações, não causas. Nada aqui autoriza afirmar que um fator *causa* a gravidade.

**Definição do rótulo.** Um acidente é **grave** quando tem pelo menos um morto ou um ferido grave (`mortos > 0` ou `feridos_graves > 0`). Os demais são **não graves**. Juntar mortos e feridos graves dá uma classe positiva com volume suficiente para modelar e corresponde aos casos que exigem atendimento de maior complexidade.

## 2. Ambiente e reprodutibilidade

Todas as etapas aleatórias usam a mesma semente (`SEMENTE = 42`). A célula abaixo imprime as versões das bibliotecas usadas nesta execução. O notebook só depende de bibliotecas que já vêm instaladas no Google Colab e no Kaggle.

In [ ]:
import io
import sys
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, accuracy_score,
                             precision_recall_curve, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMENTE = 42
np.random.seed(SEMENTE)

print("Python      ", sys.version.split()[0])
for nome, modulo in [("pandas", pd), ("numpy", np), ("scikit-learn", sklearn), ("matplotlib", matplotlib)]:
    print(f"{nome:12s}", modulo.__version__)

# Estilo único para todos os gráficos
AZUL, LARANJA, VERDE, AMARELO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.frameon": False,
})
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

## 3. Dados

| Item | Descrição |
|---|---|
| **Fonte** | Polícia Rodoviária Federal — [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf), seção "Acidentes", arquivos "Agrupados por ocorrência". |
| **Condição de uso** | A PRF publica o conjunto como dado aberto, definido na própria página como "sem restrição de licenças, patentes ou mecanismos de controle", que "qualquer pessoa pode livremente usar, reutilizar e redistribuir". Não exige cadastro nem login. |
| **Unidade de análise** | Cada linha é um acidente (uma ocorrência registrada em Boletim de Acidente de Trânsito). |
| **Período usado** | 2021 a 2025 (cinco anos completos). |
| **Formato** | Um arquivo ZIP por ano, hospedado pela PRF no Google Drive, contendo um CSV separado por `;` em codificação Latin-1. |

**Por que a partir de 2021.** Os trabalhos relacionados mais recentes usam o período 2020–2024, e o formulário de registro da PRF mudou ao longo dos anos. Cinco anos recentes dão volume suficiente e permitem separar treino, validação e teste por ano.

**Limitações conhecidas da fonte.**
- A PRF avisa que os números dos dados abertos podem divergir dos anuários estatísticos, porque os registros são revisados depois da publicação. Os arquivos podem mudar entre uma execução e outra deste notebook.
- Os campos são preenchidos pelo policial no local. Há categorias como "Ignorado" e "Não Informado", e a forma de preenchimento mudou com o tempo (mostrado na Seção 4).
- Acidentes sem acionamento da PRF não aparecem. Acidentes leves tendem a ser sub-registrados, o que infla a proporção de graves em relação à realidade.

A célula abaixo baixa os cinco arquivos direto dos links publicados na página da PRF e guarda uma cópia na pasta `dados/` para não baixar de novo.

In [ ]:
# Identificadores dos arquivos no Google Drive, copiados da página de Dados Abertos da PRF
# (linhas "Documento CSV de Acidentes <ano> (Agrupados por ocorrência)"), consultada em 16/11/2026.
ARQUIVOS_PRF = {
    2021: "12xH8LX9aN2gObR766YN3cMcuycwyCJDz",
    2022: "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-",
    2023: "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq",
    2024: "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO",
    2025: "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR",
}
PASTA_DADOS = Path("dados")
PASTA_DADOS.mkdir(exist_ok=True)


def baixar_ano(ano):
    """Baixa o ZIP de um ano (se ainda não estiver na pasta) e devolve o DataFrame."""
    destino = PASTA_DADOS / f"datatran{ano}.zip"
    if not destino.exists():
        url = f"https://drive.google.com/uc?export=download&id={ARQUIVOS_PRF[ano]}"
        requisicao = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        conteudo = urllib.request.urlopen(requisicao, timeout=120).read()
        if not conteudo.startswith(b"PK"):
            raise RuntimeError(
                f"O download de {ano} não devolveu um arquivo ZIP. Baixe manualmente em "
                "https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf "
                f"e salve como {destino}.")
        destino.write_bytes(conteudo)
    with zipfile.ZipFile(destino) as z:
        with z.open(z.namelist()[0]) as f:
            df = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
    df["ano"] = ano
    return df


bruto = pd.concat([baixar_ano(ano) for ano in ARQUIVOS_PRF], ignore_index=True)
print(f"{len(bruto):,} acidentes e {bruto.shape[1]} colunas".replace(",", "."))
bruto.groupby("ano").size().rename("acidentes").to_frame().T

### 3.1 Atributos

A tabela resume cada coluna: tipo, valores ausentes, número de valores distintos e um exemplo.

In [ ]:
resumo = pd.DataFrame({
    "tipo": bruto.dtypes.astype(str),
    "ausentes": bruto.isna().sum(),
    "distintos": bruto.nunique(),
    "exemplo": bruto.iloc[0].astype(str),
})
resumo

As colunas se dividem em três grupos, e essa divisão orienta todo o restante do trabalho:

1. **Contexto** — onde, quando e em que condições: `data_inversa`, `dia_semana`, `horario`, `uf`, `br`, `km`, `latitude`, `longitude`, `fase_dia`, `sentido_via`, `condicao_metereologica`, `tipo_pista`, `tracado_via`, `uso_solo`.
2. **Dinâmica da ocorrência** — só se conhece depois que o acidente aconteceu e foi descrito: `tipo_acidente`, `causa_acidente`, `veiculos`, `pessoas`.
3. **Desfecho** — contagem de vítimas, de onde sai o rótulo: `mortos`, `feridos_graves`, `feridos_leves`, `feridos`, `ilesos`, `ignorados`, `classificacao_acidente`.

As colunas `municipio`, `regional`, `delegacia` e `uop` são identificadores administrativos com muitas categorias e redundantes com `uf`, `br`, `km` e as coordenadas; não são usadas.

### 3.2 Rótulo e verificações de qualidade

In [ ]:
bruto["grave"] = ((bruto["mortos"] > 0) | (bruto["feridos_graves"] > 0)).astype(int)

print("Distribuição do rótulo (todos os anos):")
print(bruto["grave"].map({0: "não grave", 1: "grave"}).value_counts(normalize=True).round(3).to_string())

print("\nVerificações de qualidade:")
print("  linhas totalmente duplicadas ......", bruto.drop(columns=["ano", "grave"]).duplicated().sum())
print("  identificadores (id) repetidos ....", bruto["id"].duplicated().sum())
print("  feridos != leves + graves .........", (bruto["feridos"] != bruto["feridos_leves"] + bruto["feridos_graves"]).sum())
print("  classificacao_acidente ausente ....", bruto["classificacao_acidente"].isna().sum())
incoerente = ((bruto["classificacao_acidente"] == "Sem Vítimas") & (bruto["mortos"] + bruto["feridos"] > 0)).sum()
print("  'Sem Vítimas' com vítimas .........", incoerente)

pd.crosstab(bruto["classificacao_acidente"].fillna("(ausente)"), bruto["grave"].map({0: "não grave", 1: "grave"}))

**Leitura.** Não há linhas nem identificadores duplicados, e as contagens de vítimas são coerentes entre si. Os poucos valores ausentes ficam em `classificacao_acidente` e nas colunas administrativas, que não entram no modelo — por isso nenhuma linha precisa ser descartada.

A tabela cruzada mostra por que as colunas de desfecho não podem ser atributos: todo acidente "Com Vítimas Fatais" é grave por definição e todo "Sem Vítimas" é não grave. Usar essa coluna seria entregar a resposta ao modelo. A Seção 8 mede o tamanho desse efeito.

## 4. Preparação dos dados

A função `preparar` aplica transformações que dependem **apenas da própria linha** (conversão de tipos e criação de atributos). Nada aqui usa estatísticas calculadas sobre o conjunto, então pode ser aplicada antes da divisão sem risco de vazamento. As transformações que aprendem algo dos dados (padronização e codificação de categorias) ficam dentro do *pipeline* e são ajustadas só no treino (Seção 6).

| Decisão | Justificativa |
|---|---|
| `km`, `latitude`, `longitude` de texto para número | Vêm com vírgula decimal. |
| `horario` → hora decimal, mais `hora_sen` e `hora_cos` | A hora é cíclica: 23h e 0h são vizinhas. Seno e cosseno preservam essa vizinhança para a regressão logística. |
| `data_inversa` → `mes` e `fim_de_semana` | Captura sazonalidade e o padrão de fim de semana. |
| `tracado_via` → 12 indicadores (0/1) | O campo aceita vários valores juntos ("Reta;Declive"), o que gera mais de mil combinações. Separar em indicadores mantém a informação com 12 colunas. |
| `uso_solo` → `area_urbana` (0/1) | Na base, "Sim" indica trecho urbano. |
| `br` como categoria | O número da rodovia é um nome, não uma grandeza. |
| Categorias "Ignorado" e "Não Informado" mantidas | A falta de informação pode estar associada ao desfecho; descartar as linhas introduziria viés de seleção. |
| Sem remoção de *outliers* | Valores extremos de `veiculos` e `pessoas` são acidentes reais (engavetamentos, ônibus). As coordenadas foram conferidas contra os limites do território brasileiro. |

In [ ]:
ELEMENTOS_TRACADO = ["Reta", "Curva", "Aclive", "Declive", "Interseção de Vias", "Rotatória", "Ponte",
                     "Viaduto", "Túnel", "Retorno Regulamentado", "Desvio Temporário", "Em Obras"]
NOME_TRACADO = {e: "tracado_" + e.lower().replace(" de vias", "").replace(" regulamentado", "")
                .replace(" temporário", "").replace(" ", "_").replace("ç", "c").replace("ã", "a").replace("ú", "u")
                for e in ELEMENTOS_TRACADO}
REGIAO = {**dict.fromkeys(["AC", "AM", "AP", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MS", "MT"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}


def para_numero(coluna):
    return pd.to_numeric(coluna.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def preparar(df):
    d = pd.DataFrame(index=df.index)
    d["ano"] = df["ano"]
    d["grave"] = df["grave"]

    # --- contexto: quando
    data = pd.to_datetime(df["data_inversa"])
    hora = pd.to_datetime(df["horario"], format="%H:%M:%S")
    d["hora"] = hora.dt.hour + hora.dt.minute / 60
    d["hora_sen"] = np.sin(2 * np.pi * d["hora"] / 24)
    d["hora_cos"] = np.cos(2 * np.pi * d["hora"] / 24)
    d["mes"] = data.dt.month
    d["dia_semana"] = df["dia_semana"]
    d["fim_de_semana"] = df["dia_semana"].isin(["sábado", "domingo"]).astype(int)
    d["fase_dia"] = df["fase_dia"]

    # --- contexto: onde
    d["uf"] = df["uf"]
    d["regiao"] = df["uf"].map(REGIAO)          # usada só na análise, não como atributo
    d["br"] = "BR-" + df["br"].astype(str)
    d["km"] = para_numero(df["km"])
    d["latitude"] = para_numero(df["latitude"])
    d["longitude"] = para_numero(df["longitude"])
    d["sentido_via"] = df["sentido_via"]

    # --- contexto: condições da via
    d["condicao_metereologica"] = df["condicao_metereologica"]
    d["tipo_pista"] = df["tipo_pista"]
    d["area_urbana"] = (df["uso_solo"] == "Sim").astype(int)
    partes = df["tracado_via"].fillna("").str.split(";")
    for elemento, nome in NOME_TRACADO.items():
        d[nome] = partes.apply(lambda lista, e=elemento: int(e in lista))

    # --- dinâmica da ocorrência (conhecida só depois do acidente)
    d["tipo_acidente"] = df["tipo_acidente"]
    d["causa_acidente"] = df["causa_acidente"]
    d["veiculos"] = df["veiculos"]
    d["pessoas"] = df["pessoas"]

    # --- desfecho (usado apenas na demonstração de vazamento da Seção 8)
    d["classificacao_acidente"] = df["classificacao_acidente"].fillna("(ausente)")
    d["feridos_leves"] = df["feridos_leves"]
    d["ilesos"] = df["ilesos"]
    d["ignorados"] = df["ignorados"]
    return d


dados = preparar(bruto)
fora_do_brasil = ~(dados["latitude"].between(-34, 5.5) & dados["longitude"].between(-74.5, -32))
print("Valores ausentes após a preparação:", int(dados.isna().sum().sum()))
print("Coordenadas fora dos limites do Brasil:", int(fora_do_brasil.sum()))
dados.head(3)

## 5. Análise exploratória

Cada gráfico abaixo responde a uma pergunta que orienta uma decisão posterior. A análise exploratória usa todos os anos apenas para descrever a base; nenhuma escolha de modelo é feita olhando o ano de teste.

### 5.1 A proporção de acidentes graves é estável no tempo?

In [ ]:
por_ano = dados.groupby("ano")["grave"].agg(acidentes="size", graves="sum", proporcao="mean")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
ax1.bar(por_ano.index, por_ano["acidentes"] - por_ano["graves"], color="#b7d3f6", label="Não grave", width=0.6)
ax1.bar(por_ano.index, por_ano["graves"], bottom=por_ano["acidentes"] - por_ano["graves"], color=AZUL,
        label="Grave", width=0.6, edgecolor="white", linewidth=1.5)
ax1.set_title("Acidentes registrados por ano")
ax1.set_ylabel("Número de acidentes")
ax1.set_xlabel("Ano")
ax1.legend(loc="upper left", ncol=2)
ax1.set_ylim(0, por_ano["acidentes"].max() * 1.2)

ax2.plot(por_ano.index, por_ano["proporcao"] * 100, color=AZUL, marker="o", linewidth=2)
for ano, p in por_ano["proporcao"].items():
    ax2.annotate(f"{p*100:.1f}%", (ano, p * 100), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax2.set_ylim(0, 40)
ax2.set_title("Proporção de acidentes graves por ano")
ax2.set_ylabel("Acidentes graves (%)")
ax2.set_xlabel("Ano")
ax2.set_xticks(por_ano.index)
plt.tight_layout()
plt.show()
por_ano

**Leitura.** *(preenchido após a execução — ver texto final)*

### 5.2 Em que horários os acidentes são mais graves?

In [ ]:
dados["hora_cheia"] = dados["hora"].astype(int)
por_hora = dados.groupby("hora_cheia")["grave"].agg(acidentes="size", proporcao="mean")
media_geral = dados["grave"].mean() * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [1, 1.3]})
ax1.bar(por_hora.index, por_hora["acidentes"], color="#b7d3f6", width=0.8)
ax1.set_title("Número de acidentes por hora do dia (2021–2025)")
ax1.set_ylabel("Acidentes")
ax2.plot(por_hora.index, por_hora["proporcao"] * 100, color=AZUL, marker="o", linewidth=2, markersize=4)
ax2.axhline(media_geral, color=CINZA, linestyle="--", linewidth=1)
ax2.text(23.4, media_geral + 0.6, f"média geral {media_geral:.1f}%", ha="right", fontsize=9, color="#52514e")
ax2.set_title("Proporção de acidentes graves por hora do dia")
ax2.set_ylabel("Acidentes graves (%)")
ax2.set_xlabel("Hora do dia")
ax2.set_xticks(range(0, 24))
ax2.set_ylim(0, por_hora["proporcao"].max() * 100 * 1.2)
plt.tight_layout()
plt.show()

**Leitura.** *(preenchido após a execução)*

### 5.3 Quais condições da via e do ambiente se associam à gravidade?

In [ ]:
def taxa_por_categoria(coluna, minimo=300):
    t = dados.groupby(coluna)["grave"].agg(acidentes="size", proporcao="mean")
    return t[t["acidentes"] >= minimo].sort_values("proporcao")


paineis = [("tipo_pista", "Tipo de pista"), ("fase_dia", "Fase do dia"),
           ("condicao_metereologica", "Condição meteorológica"), ("area_urbana", "Área urbana")]
fig, eixos = plt.subplots(2, 2, figsize=(11, 6))
for ax, (coluna, titulo) in zip(eixos.ravel(), paineis):
    t = taxa_por_categoria(coluna)
    rotulos = t.index.map({0: "Não (rural)", 1: "Sim (urbana)"}) if coluna == "area_urbana" else t.index
    ax.barh(rotulos, t["proporcao"] * 100, color=AZUL, height=0.6)
    ax.axvline(media_geral, color=CINZA, linestyle="--", linewidth=1)
    for y, (p, n) in enumerate(zip(t["proporcao"], t["acidentes"])):
        ax.text(p * 100 + 0.6, y, f"{p*100:.1f}%  (n={n:,})".replace(",", "."), va="center", fontsize=8, color="#52514e")
    ax.set_xlim(0, 55)
    ax.set_title(titulo)
    ax.set_xlabel("Acidentes graves (%)")
    ax.grid(axis="y", visible=False)
fig.suptitle("Proporção de acidentes graves por condição — linha tracejada = média geral", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

**Leitura.** *(preenchido após a execução)*

### 5.4 A gravidade varia entre os estados?

In [ ]:
por_uf = dados.groupby("uf")["grave"].agg(acidentes="size", proporcao="mean").sort_values("proporcao")
cores = [LARANJA if uf == "PI" else AZUL for uf in por_uf.index]

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(por_uf.index, por_uf["proporcao"] * 100, color=cores, width=0.7)
ax.axhline(media_geral, color=CINZA, linestyle="--", linewidth=1)
ax.text(-0.4, media_geral + 0.8, f"média geral {media_geral:.1f}%", fontsize=9, color="#52514e")
ax.set_title("Proporção de acidentes graves por UF (2021–2025) — Piauí em destaque")
ax.set_ylabel("Acidentes graves (%)")
ax.set_xlabel("Unidade da Federação")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

por_regiao = dados.groupby("regiao")["grave"].agg(acidentes="size", proporcao="mean").sort_values("proporcao")
por_regiao

**Leitura.** *(preenchido após a execução)*

### 5.5 O tipo de acidente explica a gravidade? E o preenchimento da base mudou com o tempo?

O tipo de acidente não é contexto: só existe depois que o acidente acontece. Ainda assim vale olhar, porque os trabalhos relacionados o usam como atributo.

In [ ]:
t = taxa_por_categoria("tipo_acidente", minimo=1000)
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.barh(t.index, t["proporcao"] * 100, color=AZUL, height=0.65)
ax.axvline(media_geral, color=CINZA, linestyle="--", linewidth=1)
for y, (p, n) in enumerate(zip(t["proporcao"], t["acidentes"])):
    ax.text(p * 100 + 0.6, y, f"{p*100:.1f}%  (n={n:,})".replace(",", "."), va="center", fontsize=8, color="#52514e")
ax.set_xlim(0, 75)
ax.set_title("Proporção de acidentes graves por tipo de acidente")
ax.set_xlabel("Acidentes graves (%)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

mudanca = pd.DataFrame({
    "tracado_via com mais de um valor (%)": bruto.groupby("ano")["tracado_via"].apply(lambda s: s.str.contains(";").mean() * 100),
    "causas distintas": bruto.groupby("ano")["causa_acidente"].nunique(),
    "tipos distintos": bruto.groupby("ano")["tipo_acidente"].nunique(),
})
mudanca.round(1)

**Leitura.** *(preenchido após a execução)*

## 6. Protocolo experimental

**Divisão temporal.** O uso pretendido é prever acidentes *futuros* com um modelo treinado no passado. A divisão segue essa ordem:

| Conjunto | Anos | Papel |
|---|---|---|
| **Treino** | 2021, 2022, 2023 | Ajuste dos modelos e do pré-processamento. |
| **Validação** | 2024 | Escolha de hiperparâmetros e do limiar de decisão. |
| **Teste** | 2025 | Avaliação final. Só é consultado depois de todas as escolhas feitas. |

Uma divisão aleatória misturaria acidentes de todos os anos em treino e teste e poderia esconder mudanças na forma de registro (Seção 5.5). A Seção 9 compara as duas divisões para medir essa diferença.

**Prevenção de vazamento.** Três cuidados:
1. *Vazamento de rótulo*: as colunas de desfecho nunca entram como atributo no modelo principal.
2. *Vazamento de pré-processamento*: padronização e codificação ficam dentro de um `Pipeline` do scikit-learn, ajustado apenas com o treino. Validação e teste só são transformados.
3. *Vazamento por seleção*: hiperparâmetros e limiar são escolhidos na validação; o teste não participa de nenhuma decisão.

**Três conjuntos de atributos.** Para responder à pergunta de pesquisa, os mesmos modelos são treinados com três conjuntos:

| Cenário | Atributos | Quando a informação existe |
|---|---|---|
| **A — Contexto** (principal) | Local, horário, clima, tipo e traçado da pista | Antes do acidente, ou no momento do chamado. |
| **B — Contexto + dinâmica** | A + tipo e causa do acidente, nº de veículos e de pessoas | Depois do acidente, quando alguém o descreve. |
| **C — Com vazamento** | B + classificação do acidente, feridos leves, ilesos, ignorados | Só depois de contadas as vítimas, ou seja, quando o desfecho já é conhecido. |

In [ ]:
NUM_A = ["hora", "hora_sen", "hora_cos", "mes", "fim_de_semana", "km", "latitude", "longitude", "area_urbana"] \
        + list(NOME_TRACADO.values())
CAT_A = ["dia_semana", "fase_dia", "uf", "br", "sentido_via", "condicao_metereologica", "tipo_pista"]
NUM_B, CAT_B = NUM_A + ["veiculos", "pessoas"], CAT_A + ["tipo_acidente", "causa_acidente"]
NUM_C, CAT_C = NUM_B + ["feridos_leves", "ilesos", "ignorados"], CAT_B + ["classificacao_acidente"]
CENARIOS = {"A — Contexto": (NUM_A, CAT_A), "B — Contexto + dinâmica": (NUM_B, CAT_B), "C — Com vazamento": (NUM_C, CAT_C)}

treino = dados[dados["ano"] <= 2023]
validacao = dados[dados["ano"] == 2024]
teste = dados[dados["ano"] == 2025]
y_treino, y_validacao, y_teste = treino["grave"], validacao["grave"], teste["grave"]

pd.DataFrame({
    "acidentes": [len(treino), len(validacao), len(teste)],
    "graves (%)": [y_treino.mean() * 100, y_validacao.mean() * 100, y_teste.mean() * 100],
}, index=["treino (2021–2023)", "validação (2024)", "teste (2025)"]).round(1)

### 6.1 Métricas

A classe grave é minoria (cerca de 28%). Um modelo que respondesse sempre "não grave" teria cerca de 72% de acurácia sem detectar nenhum caso grave. Por isso a acurácia é reportada apenas como referência, e a avaliação se apoia em:

- **PR-AUC** (área sob a curva precisão–revocação, *average precision*): métrica principal. Resume a qualidade do ordenamento por risco sem depender de um limiar, e sua linha de base é a própria proporção de graves. É a métrica usada para escolher hiperparâmetros.
- **ROC-AUC**: probabilidade de o modelo dar nota maior a um acidente grave do que a um não grave. Linha de base 0,5.
- **Revocação** (*recall*) da classe grave: fração dos acidentes graves que o modelo aponta. É o erro mais custoso na aplicação (deixar de priorizar um caso grave).
- **Precisão**: fração dos alertas que eram de fato graves. Mede o custo de alarmes falsos.
- **F1**: média harmônica de precisão e revocação, usada para fixar o limiar de decisão na validação.

In [ ]:
def preprocessador(numericas, categoricas):
    return ColumnTransformer([
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), categoricas),
    ])


def montar(modelo, numericas, categoricas):
    """Pipeline = pré-processamento + modelo. Tudo é ajustado junto, só com os dados de treino."""
    return Pipeline([("prep", preprocessador(numericas, categoricas)), ("modelo", modelo)])


def melhor_limiar(y, prob):
    """Limiar que maximiza o F1 da classe grave. Deve ser chamado apenas com dados de validação."""
    precisao, revocacao, limiares = precision_recall_curve(y, prob)
    f1 = 2 * precisao * revocacao / np.clip(precisao + revocacao, 1e-12, None)
    return float(limiares[np.argmax(f1[:-1])])


def avaliar(y, prob, limiar):
    previsto = (prob >= limiar).astype(int)
    return {
        "PR-AUC": average_precision_score(y, prob),
        "ROC-AUC": roc_auc_score(y, prob) if len(np.unique(prob)) > 1 else 0.5,
        "F1": f1_score(y, previsto, zero_division=0),
        "Precisão": precision_score(y, previsto, zero_division=0),
        "Revocação": recall_score(y, previsto, zero_division=0),
        "Acurácia": accuracy_score(y, previsto),
    }


def prob_grave(modelo, X):
    return modelo.predict_proba(X)[:, 1]

## 7. Modelagem

**Referências (baselines).**
- *Sempre não grave*: prevê a classe majoritária. Mostra por que a acurácia engana.
- *Sempre grave*: marca tudo como grave. Tem revocação 100% e define o F1 mínimo que qualquer modelo útil precisa superar.
- **Regressão logística**: modelo linear, rápido e interpretável. É a referência "séria": se um modelo mais complexo não a supera, a complexidade não se justifica.

**Abordagem principal: Random Forest.** Conjunto de árvores de decisão treinadas em amostras diferentes dos dados. Foi escolhida porque (i) captura interações e relações não lineares que o modelo linear não vê, como "pista simples **e** noite **e** área rural"; (ii) lida bem com atributos em escalas diferentes; (iii) é o algoritmo mais usado nos trabalhos relacionados com dados da PRF, o que permite comparação direta.

**Desbalanceamento.** Os dois modelos usam `class_weight="balanced"`, que dá mais peso aos erros na classe grave em vez de criar exemplos sintéticos. É mais simples que SMOTE e não altera os dados.

**Hiperparâmetros.** Uma busca pequena em grade, avaliada na validação (2024) pela PR-AUC, no cenário A. Os valores escolhidos são reutilizados nos cenários B e C para que a única diferença entre cenários sejam os atributos.

In [ ]:
X_tr, X_va = treino[NUM_A + CAT_A], validacao[NUM_A + CAT_A]
busca = []

# --- Regressão logística: C controla a regularização (menor = mais regularizado)
for C in [0.01, 0.1, 1.0]:
    inicio = time.time()
    m = montar(LogisticRegression(C=C, class_weight="balanced", max_iter=1000), NUM_A, CAT_A).fit(X_tr, y_treino)
    busca.append({"modelo": "Regressão logística", "hiperparâmetros": f"C={C}",
                  "PR-AUC treino": average_precision_score(y_treino, prob_grave(m, X_tr)),
                  "PR-AUC validação": average_precision_score(y_validacao, prob_grave(m, X_va)),
                  "tempo (s)": time.time() - inicio, "_params": {"C": C}})

# --- Random Forest: profundidade máxima e tamanho mínimo da folha controlam a complexidade
for profundidade in [12, 20, None]:
    for folha in [5, 20]:
        inicio = time.time()
        rf = RandomForestClassifier(n_estimators=200, max_depth=profundidade, min_samples_leaf=folha,
                                    max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=SEMENTE)
        m = montar(rf, NUM_A, CAT_A).fit(X_tr, y_treino)
        busca.append({"modelo": "Random Forest", "hiperparâmetros": f"max_depth={profundidade}, min_samples_leaf={folha}",
                      "PR-AUC treino": average_precision_score(y_treino, prob_grave(m, X_tr)),
                      "PR-AUC validação": average_precision_score(y_validacao, prob_grave(m, X_va)),
                      "tempo (s)": time.time() - inicio,
                      "_params": {"max_depth": profundidade, "min_samples_leaf": folha}})

busca = pd.DataFrame(busca)
busca.drop(columns="_params").round(4)

In [ ]:
melhor = busca.loc[busca.groupby("modelo")["PR-AUC validação"].idxmax()].set_index("modelo")
PARAMS_LR = melhor.loc["Regressão logística", "_params"]
PARAMS_RF = melhor.loc["Random Forest", "_params"]
print("Regressão logística escolhida:", PARAMS_LR)
print("Random Forest escolhida:      ", PARAMS_RF)


def nova_logistica():
    return LogisticRegression(**PARAMS_LR, class_weight="balanced", max_iter=1000)


def nova_floresta(semente=SEMENTE):
    return RandomForestClassifier(n_estimators=200, **PARAMS_RF, max_features="sqrt",
                                  class_weight="balanced", n_jobs=-1, random_state=semente)

**Leitura.** *(preenchido após a execução)*

## 8. Avaliação no conjunto de teste (2025)

Com os hiperparâmetros fixados, cada modelo é treinado em 2021–2023, tem o limiar de decisão definido em 2024 e é avaliado **uma única vez** em 2025.

### 8.1 Cenário A — modelo principal e baselines

In [ ]:
resultados, modelos, limiares, probs_teste = [], {}, {}, {}


def treinar_e_avaliar(nome, estimador, cenario):
    numericas, categoricas = CENARIOS[cenario]
    colunas = numericas + categoricas
    m = montar(estimador, numericas, categoricas).fit(treino[colunas], y_treino)
    limiar = melhor_limiar(y_validacao, prob_grave(m, validacao[colunas]))
    prob = prob_grave(m, teste[colunas])
    chave = (cenario, nome)
    modelos[chave], limiares[chave], probs_teste[chave] = m, limiar, prob
    resultados.append({"cenário": cenario, "modelo": nome, "limiar": limiar, **avaliar(y_teste, prob, limiar)})


CEN_A = "A — Contexto"
# Baselines triviais: probabilidade constante; o limiar decide se tudo vira "grave" ou "não grave"
constante = np.full(len(teste), y_treino.mean())
resultados.append({"cenário": CEN_A, "modelo": "Sempre não grave", "limiar": 1.0, **avaliar(y_teste, constante, 1.0)})
resultados.append({"cenário": CEN_A, "modelo": "Sempre grave", "limiar": 0.0, **avaliar(y_teste, constante, 0.0)})
treinar_e_avaliar("Regressão logística", nova_logistica(), CEN_A)
treinar_e_avaliar("Random Forest", nova_floresta(), CEN_A)

tabela_a = pd.DataFrame(resultados).set_index("modelo").drop(columns="cenário")
tabela_a.round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

for nome, cor in [("Regressão logística", LARANJA), ("Random Forest", AZUL)]:
    prob = probs_teste[(CEN_A, nome)]
    precisao, revocacao, _ = precision_recall_curve(y_teste, prob)
    ax1.plot(revocacao, precisao, color=cor, linewidth=2, label=f"{nome} (PR-AUC {average_precision_score(y_teste, prob):.3f})")
    previsto = prob >= limiares[(CEN_A, nome)]
    ax1.scatter(recall_score(y_teste, previsto), precision_score(y_teste, previsto), color=cor, s=60, zorder=3,
                edgecolor="white", linewidth=1.5)
ax1.axhline(y_teste.mean(), color=CINZA, linestyle="--", linewidth=1, label=f"Sem modelo ({y_teste.mean():.3f})")
ax1.set_title("Curva precisão–revocação no teste (2025)")
ax1.set_xlabel("Revocação (fração dos graves encontrados)")
ax1.set_ylabel("Precisão (fração dos alertas corretos)")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.legend(loc="upper right", fontsize=8)
ax1.text(0.02, 0.03, "● = limiar escolhido na validação", transform=ax1.transAxes, fontsize=8, color="#52514e")

matriz = confusion_matrix(y_teste, probs_teste[(CEN_A, "Random Forest")] >= limiares[(CEN_A, "Random Forest")])
ax2.imshow(matriz, cmap="Blues", vmin=0, vmax=matriz.max() * 1.3)
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f"{matriz[i, j]:,}".replace(",", ".") + f"\n({matriz[i, j] / matriz[i].sum() * 100:.0f}% da linha)",
                 ha="center", va="center", fontsize=10, color="white" if matriz[i, j] > matriz.max() * 0.6 else "#0b0b0b")
ax2.set_xticks([0, 1], ["Previsto: não grave", "Previsto: grave"])
ax2.set_yticks([0, 1], ["Real: não grave", "Real: grave"])
ax2.set_title("Matriz de confusão — Random Forest, teste (2025)")
ax2.grid(False)
plt.tight_layout()
plt.show()

**Leitura.** *(preenchido após a execução)*

### 8.2 Auditoria de vazamento — cenários A, B e C

Os mesmos dois modelos, com os mesmos hiperparâmetros, são treinados agora com os atributos dos cenários B e C. A única coisa que muda é a informação disponível.

In [ ]:
for cenario in ["B — Contexto + dinâmica", "C — Com vazamento"]:
    treinar_e_avaliar("Regressão logística", nova_logistica(), cenario)
    treinar_e_avaliar("Random Forest", nova_floresta(), cenario)

tabela_cenarios = pd.DataFrame(resultados)
tabela_cenarios = tabela_cenarios[tabela_cenarios["modelo"].isin(["Regressão logística", "Random Forest"])]
tabela_cenarios = tabela_cenarios.set_index(["cenário", "modelo"]).drop(columns="limiar")
tabela_cenarios.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
nomes_cenarios = list(CENARIOS)
x = np.arange(len(nomes_cenarios))
for ax, metrica in zip(eixos, ["PR-AUC", "F1"]):
    for deslocamento, (nome, cor) in zip([-0.19, 0.19], [("Regressão logística", LARANJA), ("Random Forest", AZUL)]):
        valores = [tabela_cenarios.loc[(c, nome), metrica] for c in nomes_cenarios]
        ax.bar(x + deslocamento, valores, width=0.36, color=cor, label=nome)
        for xi, v in zip(x + deslocamento, valores):
            ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=9)
    referencia = y_teste.mean() if metrica == "PR-AUC" else tabela_a.loc["Sempre grave", "F1"]
    ax.axhline(referencia, color=CINZA, linestyle="--", linewidth=1)
    ax.text(2.45, referencia + 0.015, "sem modelo" if metrica == "PR-AUC" else "sempre grave", ha="right", fontsize=8, color="#52514e")
    ax.set_xticks(x, [c.replace(" — ", "\n") for c in nomes_cenarios])
    ax.set_ylim(0, 1.12)
    ax.set_title(f"{metrica} no teste (2025), por conjunto de atributos")
    ax.grid(axis="x", visible=False)
eixos[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**Leitura.** *(preenchido após a execução)*

## 9. Robustez

### 9.1 Divisão temporal × divisão aleatória

Muitos trabalhos dividem treino e teste por sorteio. Para medir o efeito dessa escolha, o experimento abaixo usa **exatamente as mesmas linhas** (2021–2023 e 2025) e os mesmos tamanhos de treino e teste, mudando apenas o critério de divisão. O limiar é o mesmo definido na validação de 2024.

In [ ]:
conjunto = pd.concat([treino, teste])
tr_aleatorio, te_aleatorio = train_test_split(conjunto, test_size=len(teste), stratify=conjunto["grave"], random_state=SEMENTE)
colunas_a = NUM_A + CAT_A
comparacao = []
for nome, fabrica in [("Regressão logística", nova_logistica), ("Random Forest", nova_floresta)]:
    m = montar(fabrica(), NUM_A, CAT_A).fit(tr_aleatorio[colunas_a], tr_aleatorio["grave"])
    limiar = limiares[(CEN_A, nome)]
    aleatoria = avaliar(te_aleatorio["grave"], prob_grave(m, te_aleatorio[colunas_a]), limiar)
    temporal = avaliar(y_teste, probs_teste[(CEN_A, nome)], limiar)
    for metrica in ["PR-AUC", "ROC-AUC", "F1"]:
        comparacao.append({"modelo": nome, "métrica": metrica, "divisão temporal": temporal[metrica],
                           "divisão aleatória": aleatoria[metrica], "diferença": aleatoria[metrica] - temporal[metrica]})
pd.DataFrame(comparacao).set_index(["modelo", "métrica"]).round(4)

### 9.2 Estabilidade entre sementes e incerteza da estimativa

A Random Forest tem componentes aleatórios (amostragem das linhas e dos atributos). Ela é treinada com cinco sementes diferentes para verificar se o resultado depende da semente. Além disso, o intervalo de confiança da PR-AUC no teste é estimado por *bootstrap* (reamostragem dos acidentes de 2025 com reposição, 500 vezes), o que indica se a diferença entre os modelos é maior que a variação amostral.

In [ ]:
sementes = [42, 7, 123, 2024, 99]
estabilidade = []
for s in sementes:
    if s == SEMENTE:
        prob = probs_teste[(CEN_A, "Random Forest")]
    else:
        m = montar(nova_floresta(s), NUM_A, CAT_A).fit(treino[colunas_a], y_treino)
        prob = prob_grave(m, teste[colunas_a])
    estabilidade.append({"semente": s, **avaliar(y_teste, prob, limiares[(CEN_A, "Random Forest")])})
estabilidade = pd.DataFrame(estabilidade).set_index("semente")
print("Random Forest em cinco sementes (teste 2025):")
print(estabilidade[["PR-AUC", "ROC-AUC", "F1", "Revocação"]].agg(["mean", "std", "min", "max"]).round(4).to_string())

gerador = np.random.default_rng(SEMENTE)
y_np = y_teste.to_numpy()
p_rf, p_lr = probs_teste[(CEN_A, "Random Forest")], probs_teste[(CEN_A, "Regressão logística")]
amostras = []
for _ in range(500):
    i = gerador.integers(0, len(y_np), len(y_np))
    amostras.append((average_precision_score(y_np[i], p_rf[i]), average_precision_score(y_np[i], p_lr[i])))
amostras = np.array(amostras)
diferenca = amostras[:, 0] - amostras[:, 1]
print("\nIntervalos de 95% por bootstrap (PR-AUC, teste 2025):")
print(f"  Random Forest .......... {np.percentile(amostras[:, 0], 2.5):.4f} a {np.percentile(amostras[:, 0], 97.5):.4f}")
print(f"  Regressão logística .... {np.percentile(amostras[:, 1], 2.5):.4f} a {np.percentile(amostras[:, 1], 97.5):.4f}")
print(f"  Diferença (RF − RL) .... {np.percentile(diferenca, 2.5):.4f} a {np.percentile(diferenca, 97.5):.4f}")

**Leitura.** *(preenchido após a execução)*

## 10. Análise de erros e interpretação

### 10.1 O modelo funciona igualmente bem em todas as regiões?

Um modelo com bom desempenho médio pode falhar em subgrupos. A tabela avalia a Random Forest do cenário A separadamente em cada região e no Piauí, com o mesmo limiar.

In [ ]:
CHAVE = (CEN_A, "Random Forest")
analise = teste.assign(prob=probs_teste[CHAVE], previsto=(probs_teste[CHAVE] >= limiares[CHAVE]).astype(int))


def metricas_grupo(g):
    return pd.Series({"acidentes": len(g), "graves (%)": g["grave"].mean() * 100,
                      "PR-AUC": average_precision_score(g["grave"], g["prob"]),
                      "ganho sobre a base": average_precision_score(g["grave"], g["prob"]) / g["grave"].mean(),
                      "ROC-AUC": roc_auc_score(g["grave"], g["prob"]),
                      "Revocação": recall_score(g["grave"], g["previsto"]),
                      "Precisão": precision_score(g["grave"], g["previsto"], zero_division=0),
                      "alertas (%)": g["previsto"].mean() * 100})


por_grupo = pd.concat([
    pd.DataFrame({r: metricas_grupo(g) for r, g in analise.groupby("regiao")}).T,
    pd.DataFrame({"Piauí": metricas_grupo(analise[analise["uf"] == "PI"]), "Brasil": metricas_grupo(analise)}).T,
])
por_grupo["acidentes"] = por_grupo["acidentes"].astype(int)

ordem = por_grupo.drop(index="Brasil").sort_values("Revocação")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metrica in zip([ax1, ax2], ["Revocação", "Precisão"]):
    ax.barh(ordem.index, ordem[metrica], color=[LARANJA if i == "Piauí" else AZUL for i in ordem.index], height=0.6)
    ax.axvline(por_grupo.loc["Brasil", metrica], color=CINZA, linestyle="--", linewidth=1)
    for y, v in enumerate(ordem[metrica]):
        ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_title(f"{metrica} por região — tracejado = Brasil")
    ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()
por_grupo.round(3)

**Leitura.** *(preenchido após a execução)*

### 10.2 Quais acidentes graves o modelo deixa passar?

O modelo do cenário A não conhece o tipo do acidente. Cruzar seus erros com essa coluna mostra **que tipo de caso grave o contexto sozinho não consegue antecipar**.

In [ ]:
graves = analise[analise["grave"] == 1]
por_tipo = graves.groupby("tipo_acidente")["previsto"].agg(graves="size", revocacao="mean")
por_tipo = por_tipo[por_tipo["graves"] >= 200].sort_values("revocacao")

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.barh(por_tipo.index, por_tipo["revocacao"], color=AZUL, height=0.65)
ax.axvline(por_grupo.loc["Brasil", "Revocação"], color=CINZA, linestyle="--", linewidth=1)
for y, (v, n) in enumerate(zip(por_tipo["revocacao"], por_tipo["graves"])):
    ax.text(v + 0.01, y, f"{v:.2f}  (n={n:,})".replace(",", "."), va="center", fontsize=8, color="#52514e")
ax.set_xlim(0, 1.1)
ax.set_title("Fração dos acidentes graves detectada, por tipo de acidente (teste 2025)")
ax.set_xlabel("Revocação entre os acidentes graves de cada tipo — tracejado = geral")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

# Perfil dos falsos negativos (graves não detectados) comparado ao dos graves detectados
perdidos, achados = graves[graves["previsto"] == 0], graves[graves["previsto"] == 1]
perfil = pd.DataFrame({
    "graves NÃO detectados": [len(perdidos), perdidos["area_urbana"].mean() * 100, (perdidos["tipo_pista"] == "Simples").mean() * 100,
                              (perdidos["fase_dia"] == "Plena Noite").mean() * 100, perdidos["fim_de_semana"].mean() * 100],
    "graves detectados": [len(achados), achados["area_urbana"].mean() * 100, (achados["tipo_pista"] == "Simples").mean() * 100,
                          (achados["fase_dia"] == "Plena Noite").mean() * 100, achados["fim_de_semana"].mean() * 100],
}, index=["quantidade", "em área urbana (%)", "em pista simples (%)", "em plena noite (%)", "em fim de semana (%)"])
perfil.round(1)

**Leitura.** *(preenchido após a execução)*

### 10.3 Em quais atributos o modelo se apoia?

A importância por permutação mede quanto a PR-AUC cai quando os valores de um atributo são embaralhados: se a queda é grande, o modelo depende dele. É calculada na **validação** (não no treino, para não premiar memorização, e não no teste, para mantê-lo intocado), em uma amostra de 20 mil acidentes, com cinco repetições.

In [ ]:
amostra = validacao.sample(20000, random_state=SEMENTE)
importancia = permutation_importance(modelos[CHAVE], amostra[colunas_a], amostra["grave"], scoring="average_precision",
                                     n_repeats=5, random_state=SEMENTE, n_jobs=-1)
imp = pd.DataFrame({"queda": importancia.importances_mean, "desvio": importancia.importances_std}, index=colunas_a)
imp = imp.sort_values("queda").tail(15)

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh(imp.index, imp["queda"], xerr=imp["desvio"], color=AZUL, height=0.65, ecolor="#52514e", capsize=2)
ax.set_title("Importância por permutação — Random Forest, cenário A (validação 2024)")
ax.set_xlabel("Queda na PR-AUC quando o atributo é embaralhado (média de 5 repetições, barra = desvio padrão)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Leitura.** *(preenchido após a execução)*

### 10.4 As probabilidades do modelo podem ser lidas como risco?

In [ ]:
faixas = pd.cut(analise["prob"], bins=np.linspace(0, 1, 11))
calibracao = analise.groupby(faixas, observed=True).agg(previsto_medio=("prob", "mean"), observado=("grave", "mean"), acidentes=("grave", "size"))

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot([0, 1], [0, 1], color=CINZA, linestyle="--", linewidth=1, label="Calibração perfeita")
ax.plot(calibracao["previsto_medio"], calibracao["observado"], color=AZUL, marker="o", linewidth=2, label="Random Forest (cenário A)")
ax.set_title("Calibração no teste (2025)")
ax.set_xlabel("Probabilidade média prevista de ser grave")
ax.set_ylabel("Proporção observada de graves")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()
calibracao.round(3)

**Leitura.** *(preenchido após a execução)*

## 11. Comparação com trabalhos relacionados

*(preenchido após a execução)*

## 12. Conclusões

*(preenchido após a execução)*

## 13. Transparência

**Dados.** Polícia Rodoviária Federal — Dados Abertos, Acidentes agrupados por ocorrência, 2021–2025: <https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf>

**Notebook.** *PREENCHER: link público do notebook no Colab ou Kaggle.*

**Uso de modelos de linguagem (LLMs).** *PREENCHER com a sua declaração. Rascunho factual:* Foi utilizado o assistente Claude (Anthropic), via Claude Code, para: leitura do enunciado e planejamento das etapas; busca inicial de trabalhos relacionados; geração do código e da primeira versão dos textos deste notebook; e depuração. O autor revisou e executou todas as células, conferiu os resultados contra os dados e leu os trabalhos citados nas fontes originais. Respostas do assistente não foram usadas como fonte científica.

## Referências

*PREENCHER com a referência completa de cada trabalho, conferida no original.*

1. Polícia Rodoviária Federal. *Dados Abertos da PRF — Acidentes*. Disponível em: <https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf>. Acesso em: 16 nov. 2026.
2. Gabardo et al. *Highway to... Determining Fatal Outcomes in Traffic Accidents Based on Police Reports*. BRACIS 2025. <https://sol.sbc.org.br/index.php/bracis/article/view/40861>
3. Estudo sobre gravidade de acidentes nas rodovias federais do Rio de Janeiro (2020–2023). REIC/SBC, 2025. <https://journals-sol.sbc.org.br/index.php/reic/article/download/6274/3563/33284>
4. *Predição de acidentes rodoviários em Santa Catarina: impactos de aperfeiçoamentos dos dados*. ERBD/SBC, 2025. <https://sol.sbc.org.br/index.php/erbd/article/view/35405>
5. Pedregosa et al. *Scikit-learn: Machine Learning in Python*. JMLR 12, 2011.
6. Breiman, L. *Random Forests*. Machine Learning 45, 2001.